# Black-Gold — a quantitative teardown 🔬
### The predictive regression with a t-stat · timing vs buy-and-hold · sub-period slopes

![Signal: None](https://img.shields.io/badge/Signal-None-c0392b?style=flat-square)
![Tradability: Mirage](https://img.shields.io/badge/Tradability-Mirage-c0392b?style=flat-square)
![Replicates out of sample?: Busted](https://img.shields.io/badge/Replicates_out_of_sample%3F-Busted-8b949e?style=flat-square)

The deep companion to the [notebook for the curious](01_for_the_curious.ipynb). We test oil→equity predictability on all tradable data and find none.

> ⚠️ **Not investment advice.** WTI (CL=F) + S&P 500 (^GSPC) monthly, 2000–2026 (Yahoo). CL=F begins 2000, so Driesprong's 1973–2003 window isn't testable here — but the value is out-of-sample survival. Sources in [`docs/references.md`](../docs/references.md).

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath("../../.."))  # repo root (quantlab/)
sys.path.insert(0, os.path.abspath(".."))        # study package (black_gold/)
%matplotlib inline
import matplotlib.pyplot as plt
plt.rcParams["figure.figsize"] = (10, 5.5); plt.rcParams["axes.grid"] = True
import numpy as np, pandas as pd
from black_gold import data, strategy as st
d = data.fetch_pair()                          # cache-first; built by examples/verify.py --fetch
reg = st.predict_regression(d["oil"], d["eq"])
timing, bh = st.oil_timing(d["oil"], d["eq"]), st.buy_hold(d["eq"])


## Verdict, up front

| Axis | Stamp | Why |
|---|---|---|
| Signal | **None** | slope +0.017, t +0.6, wrong sign |
| Tradability | **Mirage** | timing Sharpe 0.26 < buy-and-hold 0.42 |
| Replicates OOS? | **Busted** | insignificant in 2000–2008 and 2009-on |

> 💡 *In plain words:* a documented predictor that vanished out of sample.

## 1 · The claim, steelmanned

- **H₁:** the slope of equity_t on oil_(t−1) is negative and significant.
- **H₂:** the oil-timing rule beats buy-and-hold.
- **H₃:** it holds across sub-periods.

## 2 · So what? — what rides on each

If H₁/H₂ hold, a public price forecasts the market — a tradable cross-asset edge. If they fail OOS, it's a 1973–2003 in-sample artifact.

## 3 · How we'd know — the protocol

OLS equity_t ~ oil_(t−1) with an analytic t-stat → timing vs buy-and-hold → 2000–2008 / 2009-on slope split.

## 4 · The teardown

### 4.1 The predictive regression

In [2]:
print({k:(round(v,3) if isinstance(v,float) else v) for k,v in reg.items()})
print(f"slope {reg['slope']:+.3f} (Driesprong: negative), t = {reg['tstat']:+.2f} (need |t|>2)")

{'slope': 0.017, 'r': 0.043, 'tstat': 0.642, 'n': 220}
slope +0.017 (Driesprong: negative), t = +0.64 (need |t|>2)


> 💡 *In plain words:* t 0.6 and a positive slope — no relationship, and the wrong way. **H₁ rejected.**

### 4.2 Timing vs buy-and-hold

In [3]:
display(pd.DataFrame({'oil timing':st.summary(timing),'buy & hold':st.summary(bh)}).T[['cagr','sharpe','vol_ann','max_drawdown']].round(3))
print('time in market:', f"{st.time_in_market(d['oil']):.0%}")

,cagr,sharpe,vol_ann,max_drawdown
oil timing,0.022,0.259,0.109,-0.379
buy & hold,0.051,0.418,0.146,-0.411


time in market: 46%


> 💡 *In plain words:* lower return, lower Sharpe, half the time in cash. **H₂ rejected.**

### 4.3 Sub-period slopes

In [4]:
for lab,sl in [('2000-2008',d[d.index.year<=2008]),('2009-on',d[d.index.year>=2009])]:
    rr=st.predict_regression(sl['oil'],sl['eq']); print(f"{lab}: slope {rr['slope']:+.3f}  t={rr['tstat']:+.2f}  n={rr['n']}")

2000-2008: slope +0.017  t=+0.32  n=71
2009-on: slope +0.005  t=+0.16  n=148


> 💡 *In plain words:* insignificant and positive in both halves. **H₃ rejected** — never there in the tradable era.

## 5 · The verdict

H₁, H₂, H₃ all rejected → Signal `NONE`, Tradability `MIRAGE`, out-of-sample replication `BUSTED`.

## 6 · Could you trade it?

No edge to trade. The oil timer underperforms buy-and-hold and adds whipsaw. If oil matters for equities, it's contemporaneous and already priced — not a one-month-ahead signal.

## 7 · Going further

Forks: (a) a longer WTI spot series (pre-2000) to check the original window; (b) oil *shocks* (structural decomposition, Kilian 2009) vs raw returns; (c) sector-level (energy vs the rest) where a contemporaneous oil beta is real. Backlog: [`docs/pwb_strategies_inventory.md`](../../../docs/pwb_strategies_inventory.md).